# Lab 5 — Three detectors, one dataset
### EP0418 · Topic 5 · 2 practical hours

**You need:** `ep0418.py` and `smart_building_sensors.csv`.

---
Today you build three anomaly detectors of rising sophistication on the same data
and compare **what each one catches and what each one misses**. The comparison is
the lab; the code is not.

## Putting the assistant's code into a TODO cell

A TODO cell has a line ending in `...`. Replace **only that line** with the
assistant's code, and keep the name on the left (`s`, `clean`, `H`). Delete any
`import` or `read_csv` lines it adds; the data is already loaded. If it used a
different name for the result, add one line at the end to rename it, e.g.
`s = hourly_df`.

---
## How every lab in this module works

You will **not** write this code from scratch. Ask your AI assistant to write it.
Your job is the part it cannot do: deciding what the right answer looks like,
reading what it gave you, and proving it did what you asked.

At the foot of each task is a **verification contract** — a few `check.*` lines.
**You may not edit them.** The AI can write any implementation it likes; it cannot
satisfy these by accident.

If a check fails, read the message. It tells you what was found and what to try
next. Failing checks are the normal state of a lab in progress, not a sign you
are behind.

> **First: File → Save a copy in Drive, and work in your copy.** This page opened from GitHub, and changes here are not saved. Reopen your copy from Google Drive next time.

## Set up Gemini for this notebook (once, 30 seconds)

Open Gemini (the spark icon at the bottom of the notebook). Then:

1. In its chat box, click the **sliders icon** next to **+** and choose **Configure**. The Notebook settings window opens.
2. Click **Add new custom instruction**. Name it **EP0418**, paste the text below into Instructions, click **Add to notebook**, then **Apply**.
3. Click the sliders icon again and choose **EP0418**, so the tick moves from Default to EP0418. That is the mode you work in.

Colab keeps the instruction with this notebook. Do the same once in each new notebook; the text is always in this cell.

```
I am new to coding. Explain any code you write in plain words: what it does to my data.
Before writing code for a bigger task, tell me your plan in plain words.
Never delete or drop rows unless I ask. If a step would remove rows, say so first.
After changing the data, show the row count before and after, and the smallest and largest values.
Codes such as -999 mean a missing reading. Point them out; do not treat them as real numbers.
Do not change any line that starts with check. Those are the lab's locked checks.
If you are not sure, say so. Do not guess.
```

**Why these seven lines.** Each one stops a mistake that AI assistants make with data, and that you would not notice until a check failed:

| Line | The mistake it prevents |
|---|---|
| Explain in plain words | Code you cannot account for. You will be asked what each step does to your data. |
| Plan first | Ten lines of code for the wrong task. The plan is quicker to correct than the code. |
| Never drop rows unless asked | `dropna()` quietly removing a third of a sensor file. |
| Row count and range before and after | The only cheap way to see what a step really did. |
| -999 is missing, not a number | An average of humidity that comes out negative. |
| Never touch `check.` lines | A test rewritten so that it passes. The checks are the contract. |
| Say so if unsure | A confident guess that reads exactly like a fact. |

**See it working (1 minute).** With EP0418 selected, ask Gemini:

> Write code to remove the rows of df where humidity_pct is -999.

It should warn you that this removes rows before it writes anything, treat -999 as a missing reading rather than a value, and explain the code in plain words. Now switch the tick to **Default**, ask the same thing in a new chat, and compare. Switch back to **EP0418**.

**Make it yours.** Click the sliders icon, choose Configure, and open EP0418 with the pencil icon. Add one line of your own, for example *Keep answers under ten lines* or *Always name the column you are using*, and Apply. Ask the same question again and see your line take effect. The seven course lines stay; yours go underneath.

In [ ]:
import os, urllib.request
REPO = "https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/"


def fetch(*names):
    """Download any course file that is not beside the notebook yet."""
    for name in names:
        if os.path.exists(name):
            continue
        try:
            urllib.request.urlretrieve(REPO + name, name)
            print("fetched", name)
        except Exception:
            print(f"Could not download {name}. Upload it with the folder icon on the left.")
            from google.colab import files
            files.upload()


fetch("ep0418.py")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from ep0418 import check, show


def filled(*values):
    """True once every TODO slot above has been replaced. Otherwise says so."""
    if any(v is ... or v is None for v in values):
        print("A TODO above is still '...'. Fill it in, then run this cell again.")
        return False
    return True

CSV = "smart_building_sensors.csv"
fetch(CSV)

raw = pd.read_csv(CSV, parse_dates=["timestamp"])

# Start from clean data — reuse your Lab 4 cleaning here
df = raw.copy()
df.loc[df.humidity_pct == -999, "humidity_pct"] = np.nan
df = df.drop_duplicates(subset="timestamp", keep="first").sort_values("timestamp")
print(f"{len(df):,} rows ready")

# Placeholders so later cells give a readable message, not a NameError.
alert_fixed = alert_z = alert_ctx = z = stuck = None

## Detector 1 — Fixed threshold

The baseline. One line, and what most real buildings actually run.

In [ ]:
# TODO(you): flag every reading where co2_ppm > 1000 as `alert_fixed`
# alert_fixed = ...

if alert_fixed is None:
    print("Fill in alert_fixed above, then run this cell again.")
else:
    print(f"alerts: {alert_fixed.sum():,} of {len(df):,}  ({alert_fixed.mean():.1%})")
    print(f"fired while room EMPTY: {(alert_fixed & (df.occupancy == 0)).sum()}")
    print()
    print("alerts per week — watch what happens over time:")
    print(alert_fixed.groupby(df.timestamp.dt.isocalendar().week).sum().to_string())
    show.flags(df, alert_fixed, col="co2_ppm", title="Detector 1: fixed threshold at 1000 ppm")

**Expected output** *(computed from the data file when this notebook was built)*: 1,873 alerts, 20 of them while the room was empty; the weekly count climbs from about 154 to about 450 as the drift sets in.

**What you should see:** flags all along the series, and far more of them in the
last three weeks, where the whole line sits higher. The room did not get busier;
the sensor drifted. A fixed threshold cannot tell the difference, which is why the
weekly count climbs.

### Why is it getting worse?

The alert count climbs sharply in the final weeks. **The room did not change.**

A fixed threshold cannot tell the difference between "the air got worse" and "the
sensor drifted". Yours should.

## Detector 2 — Rolling z-score

Compare each reading against its own recent past instead of a fixed number.

> Compute a rolling mean and rolling standard deviation of `co2_ppm` over a
> 24-hour window, then a z-score for each reading. Flag readings with |z| > 3.

In [ ]:
# TODO(you): build `z` and `alert_z`
z = ...
alert_z = ...

if filled(z, alert_z):
    print(f"z-score alerts: {alert_z.sum():,}")
    print(f"fixed alerts  : {alert_fixed.sum():,}")
    show.flags(df, alert_z, col="co2_ppm", title="Detector 2: rolling z-score")

**Expected output** *(computed from the data file when this notebook was built)*: a few hundred alerts (about 416 with a 24-hour window and a cut-off of 3), far fewer than the fixed threshold. Thousands means the window is too short; see the read-before-you-run task at the end.

**What you should see:** far fewer flags, and they sit on sudden jumps rather than
on everything above a line. The drift has largely disappeared, because a rolling
window compares each reading with its recent past, and the recent past drifted
too. Some flags now fall on ordinary morning rises: the window does not know that
nine o'clock is always like that.

## Detector 3 — Contextual z-score

> 28°C at 3am in an empty room is strange. 28°C at 2pm with thirty people is
> Tuesday.

Compute the z-score **within each hour-of-day bucket**, so each reading is
compared against readings from similar circumstances.

In [ ]:
# TODO(you): build `alert_ctx` — z-score computed within hour-of-day groups
alert_ctx = ...

if filled(alert_ctx):
    print(f"contextual alerts: {alert_ctx.sum():,}")
    show.flags(df, alert_ctx, col="co2_ppm", title="Detector 3: z-score within hour-of-day groups")

**Expected output** *(computed from the data file when this notebook was built)*: the fewest of the three: under twenty with a cut-off of 3 (about 7 on this file). Hundreds means the groups were not formed by hour of day.

**What you should see:** the fewest flags of the three, and the morning rises are
gone from them. What is left is readings that are unusual *for that time of day*,
which is the only kind of unusual a person in the room would call an event.

### See which alarms disappeared

Counts alone do not show *which* alerts each detector drops. Plot one week.

In [ ]:
if filled(alert_fixed, alert_z, alert_ctx):
    week = (df.timestamp >= "2026-06-15") & (df.timestamp < "2026-06-22")
    fig, ax = plt.subplots(figsize=(11, 3.5))
    ax.plot(df.timestamp[week], df.co2_ppm[week], color="0.6", lw=0.8)
    for name, mask, y in [("fixed", alert_fixed, 1280), ("rolling z", alert_z, 1240),
                          ("contextual z", alert_ctx, 1200)]:
        m = week & mask
        ax.plot(df.timestamp[m], [y] * int(m.sum()), "|", ms=8, label=f"{name} ({int(m.sum())})")
    ax.axhline(1000, ls="--", color="0.4", lw=0.7)
    ax.set_ylabel("co2_ppm"); ax.legend(loc="lower left"); plt.show()

## Task 4 — Compare them honestly

Fill this in from your own numbers:

| Detector | Alerts fired | Fired when room empty | Fooled by the drift? | Catches the stuck sensor? |
|---|---|---|---|---|
| Fixed threshold | | | | |
| Rolling z-score | | | | |
| Contextual z-score | | | | |

**None of the three catches the stuck sensor.** Work out why, then build the one
statistic that does.

The stuck sensor from Lab 4 is in **`temp_c`**, not CO2: on 14 May it reports
23.87 °C for fourteen hours. Build the detector on that channel.

> Compute the rolling standard deviation of `temp_c` over a 1-hour window
> (12 readings). Flag readings where it is exactly zero as `stuck`.

In [ ]:
# TODO(you): a stuck sensor produces no outliers — it produces no VARIATION.
#            Build a detector for it on temp_c.
stuck = ...

if filled(stuck):
    print(f"stuck-sensor readings flagged: {int(stuck.sum())}")
    show.flags(df, stuck, col="temp_c", title="The stuck-sensor detector on temperature")

# ---- verification contract — do not edit ----
check.flagged_at_least(stuck, minimum=100, what="stuck readings")
check.report()

**Expected output** *(computed from the data file when this notebook was built)*: about 157 readings flagged, all on 14 May (the exact count depends on how long you require the line to stay flat).

**What you should see:** one solid block of flags on 14 May, fourteen hours long,
on a line that is otherwise wobbling normally. If your flags are scattered across
the whole series, the detector is reacting to small natural pauses; lengthen the
window it needs to be flat for.

---
## Read before you run

An assistant was asked for a rolling z-score of CO2 over a 24-hour window. It wrote:

**Do not run it yet.** Read it, answer the two questions below,
and only then remove the `#` and run it to see whether you were right.

The comment says 24 hours. Each row is five minutes apart. How long is the window really?

Write down the correct argument for a 24-hour window on this data.

<details><summary>Hint 1: where to look</summary>

`rolling(24)` counts rows, not hours. How many rows make one hour at five-minute spacing?

</details>
<details><summary>Hint 2: closer to the answer</summary>

Twelve rows an hour, so 24 rows is two hours. A 24-hour window is 288 rows, or `rolling("24h")` on a time index.

</details>

**After you have written your answer, and not before:** select the cell, click the
pencil-and-spark icon at its top right and choose **Explain code**. Did Gemini notice
the fault? It often does not. That is the point: an explanation that reads well is
not evidence the code is right.

**To fix it:** choose **Transform code** on the same cell and say what to change, in
plain words. Colab shows the change as a diff before it is applied. Read the diff. If
it touches a `check.` line, reject it.

In [ ]:
# --- AI-generated. Read it first. ---
# r = df.co2_ppm.rolling(24)          # 24-hour window
# z = (df.co2_ppm - r.mean()) / r.std()
# alert_z = z.abs() > 3

**What is wrong with it:**

**What it would produce, and what it should produce:**

---
## Quiz yourself before you leave (5 minutes)

The checks tell you whether the code is right. They do not tell you whether
**you** know what it did. Stay in EP0418 mode and paste this into the Gemini chat:

> Quiz me on this lab. Ask five short questions, one at a time, in plain words, on these ideas: what a z-score flags and what it misses; why a contextual detector raises fewer false alarms; why a stuck sensor produces no outliers; the difference between a false alarm and a missed event; why the choice between them belongs to the person who pays for the mistake. Take the examples from the code and outputs in this notebook. Wait for my answer. Then say in one sentence whether I am right, and why. Do not give the next question until I answer. After the fifth, give my score out of 5 and the one idea I should revise, in one line.

Gemini asks in the chat panel; you answer in the chat box; it marks you and asks
the next. Ignore the suggestion chips under its messages. Answer in your own
words, without looking back until it has marked you.

Then write your score and the idea to revise below. Your tutor asks for them in
the plenary.

**Quiz score:** ___ / 5

**The idea I should revise, in my own words:**

---
## Before you leave

Below:

- **Mistake caught:** what the assistant got wrong today and how you saw it.
- **Decision:** which detector you would actually deploy, and what it costs you.
- Which error is worse in **your** track — a false alarm, or a missed detection?

Add a **Decisions** line: the detector and threshold you chose, and why.

**Mistake caught:** *(what I expected · what was there · how I found it)*

**Decision:** *(what I decided · why · the evidence)*